# Notebook 05 — Validation Against Test Set (2023–2026)
**Layer:** Quality assurance  
**Inputs:** `hdb_feature_test_20260403.csv` · full pipeline from NB01–NB04  
**Outputs:** Intent accuracy · RMSE% · hallucination audit report

## 5.1 Load test set

In [7]:
import json
import re
import numpy as np
import faiss
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize
from neo4j import GraphDatabase
import ollama

# Local ollama — no API key, no rate limits, no internet required
OLLAMA_MODEL_Chat = "gemma3"
OLLAMA_MODEL = "nomic-embed-text"

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
CSV_PATH    = FEATURE_DIR / "hdb_feature_table_20260412.csv"

INDEX_DIR   = Path("03_vector_index")
META_PATH   = FEATURE_DIR / "feature_metadata_20260412.json"

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "pass@Word123"   # <-- update

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# Load EMBED_DIM from NB02 config
with open(INDEX_DIR / "embed_config.json") as f:
    cfg = json.load(f)
EMBED_DIM = cfg["embed_dim"]
print(f"Embedding method : {cfg['method']}")
print(f"Model chat" + OLLAMA_MODEL_Chat)
print(f"EMBED_DIM        : {EMBED_DIM}")

# Load FAISS indexes and metadata
index_pre  = faiss.read_index(str(INDEX_DIR / "index_pre2023.faiss"))
index_post = faiss.read_index(str(INDEX_DIR / "index_post2023.faiss"))
meta_pre   = pd.read_parquet(INDEX_DIR / "meta_pre2023.parquet")
meta_post  = pd.read_parquet(INDEX_DIR / "meta_post2023.parquet")

with open(META_PATH) as f:
    feature_meta = json.load(f)

# Price drift constants from README section 3
TRAIN_MEAN_PRICE = 468788
TEST_MEAN_PRICE  = 614711
TEMPORAL_SPLIT   = 2023

print(f"pre-2023  index  : {index_pre.ntotal:,} vectors")
print(f"post-2023 index  : {index_post.ntotal:,} vectors")
print(f"ollama model     : {OLLAMA_MODEL}")

def call_ollama(prompt: str, system: str = "", fmt: str = "") -> str:
    """
    Call gemma3 via local ollama. No API key, no rate limits, no internet.
    - prompt : user message
    - system : optional system instruction
    - fmt    : pass "json" to request JSON-only output
    Returns the response text string.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": OLLAMA_MODEL_Chat, "messages": messages}
    if fmt == "json":
        kwargs["format"] = "json"

    response = ollama.chat(**kwargs)
    return response.message.content.strip()


VALID_TOWNS = [
    "ANG MO KIO","BEDOK","BISHAN","BUKIT BATOK","BUKIT MERAH","BUKIT PANJANG",
    "BUKIT TIMAH","CENTRAL AREA","CHOA CHU KANG","CLEMENTI","GEYLANG","HOUGANG",
    "JURONG EAST","JURONG WEST","KALLANG/WHAMPOA","MARINE PARADE","PASIR RIS",
    "PUNGGOL","QUEENSTOWN","SEMBAWANG","SENGKANG","SERANGOON","TAMPINES",
    "TOA PAYOH","WOODLANDS","YISHUN",
]
VALID_FLAT_TYPES = [
    "1 ROOM","2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE","MULTI-GENERATION"
]

CLASSIFIER_SYSTEM = """You are a query classifier for an HDB flat database system.
Classify into ONE of: PRICE_ESTIMATION, NEIGHBOURHOOD, SCHOOL_CATCHMENT,
INVESTMENT_TEMPORAL, LEASE_ADVISORY.
Extract slots: town, flat_type, room_count, budget_sgd_min, budget_sgd_max,
year_min, year_max, lease_years_max (null if not mentioned).
Respond ONLY with valid JSON:
{"intent": "...", "slots": {"town": null, "flat_type": null, "room_count": null,
"budget_sgd_min": null, "budget_sgd_max": null, "year_min": null,
"year_max": null, "lease_years_max": null}}
"""

CYPHER_TEMPLATES = {
    "PRICE_ESTIMATION": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town) AND ($flat_type IS NULL OR f.flat_type=$flat_type)
  AND ($room_count IS NULL OR f.room_count=$room_count)
  AND ($budget_sgd_min IS NULL OR f.resale_price>=$budget_sgd_min)
  AND ($budget_sgd_max IS NULL OR f.resale_price<=$budget_sgd_max)
RETURN percentileCont(f.resale_price,0.5) AS median_price,
       avg(f.resale_price) AS avg_price, stDev(f.resale_price) AS std_price,
       count(f) AS tx_count, min(f.transaction_year) AS year_min,
       max(f.transaction_year) AS year_max""",
    "NEIGHBOURHOOD": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town) WHERE ($town IS NULL OR t.name=$town)
RETURN t.name AS town, avg(f.dist_to_mrt_m) AS avg_mrt_dist_m,
       avg(f.dist_to_foodcourt_m) AS avg_foodcourt_dist_m,
       avg(f.mall_count_3km) AS avg_mall_count_3km, count(f) AS flat_count""",
    "SCHOOL_CATCHMENT": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($budget_sgd_max IS NULL OR f.resale_price<=$budget_sgd_max)
RETURN t.name AS town,
       avg(f.primary_school_quality_1km_weighted) AS avg_school_quality,
       avg(f.school_count_1km) AS avg_schools_in_1km,
       avg(f.resale_price) AS avg_price, count(f) AS flat_count
ORDER BY avg_school_quality DESC""",
    "INVESTMENT_TEMPORAL": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($year_min IS NULL OR f.transaction_year>=$year_min)
  AND ($year_max IS NULL OR f.transaction_year<=$year_max)
RETURN t.name AS town, f.transaction_year AS year,
       avg(f.resale_price) AS avg_price, count(f) AS tx_count
ORDER BY t.name, f.transaction_year""",
    "LEASE_ADVISORY": """
MATCH (f:Flat)-[:IN_TOWN]->(t:Town)
WHERE ($town IS NULL OR t.name=$town)
  AND ($lease_years_max IS NULL OR f.lease_remaining_years<=$lease_years_max)
RETURN CASE WHEN f.lease_remaining_years<50 THEN '<50 years'
            WHEN f.lease_remaining_years<70 THEN '50-69 years'
            ELSE '70+ years' END AS lease_band,
       avg(f.resale_price) AS avg_price, count(f) AS tx_count
ORDER BY lease_band"""
}

def classify_query(user_query: str) -> dict:
    prompt = "Classify this query and extract slots:\n\n" + user_query
    raw    = call_ollama(prompt, system=CLASSIFIER_SYSTEM, fmt="json")
    return json.loads(re.sub(r"```json|```", "", raw).strip())

def slots_to_params(slots: dict) -> dict:
    town      = slots.get("town")
    flat_type = slots.get("flat_type")
    if town and town.upper() not in VALID_TOWNS:               town = None
    if flat_type and flat_type.upper() not in VALID_FLAT_TYPES: flat_type = None
    return {
        "town": town.upper() if town else None,
        "flat_type": flat_type.upper() if flat_type else None,
        "room_count": slots.get("room_count"),
        "budget_sgd_min": slots.get("budget_sgd_min"),
        "budget_sgd_max": slots.get("budget_sgd_max"),
        "year_min": slots.get("year_min"),
        "year_max": slots.get("year_max"),
        "lease_years_max": slots.get("lease_years_max"),
    }

print("Routing helpers defined.")

def embed_query(query_text: str) -> np.ndarray:
    # Embed a single query string using ollama gemma3
    resp = ollama.embed(model=OLLAMA_MODEL, input=[query_text])
    vec  = np.array(resp.embeddings[0], dtype=np.float32).reshape(1, -1)
    return normalize(vec, norm="l2").astype(np.float32)   # (1, EMBED_DIM)


def is_recent_query(slots: dict) -> bool:
    year_min = slots.get("year_min")
    year_max = slots.get("year_max")
    if year_min and year_min >= TEMPORAL_SPLIT: return True
    if year_max and year_max >= TEMPORAL_SPLIT: return True
    return False


def vector_retrieve(query_text: str, is_recent: bool, top_k: int = 20) -> pd.DataFrame:
    q_vec = embed_query(query_text)
    faiss.normalize_L2(q_vec)
    if is_recent:
        D, I = index_post.search(q_vec, top_k);  meta = meta_post
    else:
        D, I = index_pre.search(q_vec, top_k);   meta = meta_pre
    hits = meta.iloc[I[0]].copy()
    hits["similarity_score"] = D[0]
    return hits.reset_index(drop=True)


print("embed_query() and vector_retrieve() defined.")

FACTOR_MAP = {
    "PRICE_ESTIMATION":   ["level_mid","lease_remaining_years","floor_area_sqm","room_count","dist_to_mrt_m"],
    "NEIGHBOURHOOD":      ["dist_to_mrt_m","dist_to_highway_m","dist_to_foodcourt_m","mall_count_3km"],
    "SCHOOL_CATCHMENT":   ["dist_to_nearest_school_m","school_count_1km","primary_school_quality_1km_weighted"],
    "INVESTMENT_TEMPORAL":["transaction_year","resale_price"],
    "LEASE_ADVISORY":     ["lease_remaining_years","resale_price"],
}

def assemble_context(user_query, intent, cypher_records, vector_hits, is_recent):
    ctx = ["=== GRAPH AGGREGATES (from Neo4j) ===",
           json.dumps(cypher_records[:10], indent=2, default=str),
           "\n=== COMPARABLE TRANSACTIONS (top-5) ==="]
    COMP = ["address_key","town","flat_type","floor_area_sqm",
            "level_mid","lease_remaining_years","resale_price","transaction_year"]
    ctx.append(vector_hits.head(5)[COMP].to_string(index=False))
    defs = [f"  {f}: {feature_meta.get('features',{}).get(f,{}).get('description','')}"
            for f in FACTOR_MAP.get(intent,[]) if f in feature_meta.get("features",{})]
    if defs:
        ctx.append("\n=== FEATURE DEFINITIONS ==="); ctx.extend(defs)
    if is_recent:
        ctx.append(
            f"\n=== TEMPORAL CONTEXT ===\n"
            f"Post-2023 data. Mean price ${TEST_MEAN_PRICE:,} vs "
            f"pre-2023 ${TRAIN_MEAN_PRICE:,} (+31%). Include caveat."
        )
    return "\n".join(ctx)


GENERATION_SYSTEM = """You are an expert HDB property advisor for Singapore.
RULES:
1. Answer ONLY using GRAPH AGGREGATES and COMPARABLE TRANSACTIONS provided.
2. Do NOT introduce any price figures, names or locations not in the context.
3. Cite the transaction_year range of comparables used.
4. Output ONLY valid JSON, no markdown, no explanation.
SCHEMA:
{"estimate_sgd": integer_or_null, "low_sgd": integer_or_null,
 "high_sgd": integer_or_null, "basis_count": integer,
 "key_factors": [{"factor_name": str, "value": str, "impact": str}],
 "caveats": [str], "comparable_years_range": str, "narrative": str}
"""

def generate_answer(user_query: str, context: str) -> dict:
    prompt = context + "\n\nUSER QUERY: " + user_query
    raw    = call_ollama(prompt, system=GENERATION_SYSTEM, fmt="json")
    return json.loads(re.sub(r"```json|```", "", raw).strip())

print("assemble_context() and generate_answer() defined.")

#OLLAMA_MODEL        = "nomic-embed-text"
def run_pipeline(user_query: str) -> dict:
    classified    = classify_query(user_query)
    intent        = classified["intent"]
    slots         = classified["slots"]
    params        = slots_to_params(slots)
    recent        = is_recent_query(slots)

    with driver.session() as session:
        cypher_records = [dict(r) for r in
                          session.run(CYPHER_TEMPLATES[intent], **params)]

    vector_hits = vector_retrieve(user_query, is_recent=recent, top_k=20)
    context     = assemble_context(user_query, intent, cypher_records,
                                   vector_hits, recent)
    answer      = generate_answer(user_query, context)

    return {"query": user_query, "intent": intent, "slots": slots,
            "is_recent": recent, "cypher_rows": len(cypher_records),
            "vector_hits": len(vector_hits), "answer": answer}

Embedding method : ollama_gemma3
Model chatgemma3
EMBED_DIM        : 768
pre-2023  index  : 178,589 vectors
post-2023 index  : 82,110 vectors
ollama model     : nomic-embed-text
Routing helpers defined.
embed_query() and vector_retrieve() defined.
assemble_context() and generate_answer() defined.


In [8]:
import pandas as pd
import numpy as np
import json, re
from pathlib import Path

ROOT = Path('D:\Master Degree\Projects\Transparent_AI\PropertyLens')
FEATURE_DIR = ROOT / '02_feature_layer' / 'training' / 'outputs'
TEST_CSV    = FEATURE_DIR / "hdb_feature_table_20260412.csv"
#TEST_CSV = FEATURE_DIR / "hdb_feature_test_20260403.csv"

# NEVER load hdb_feature_train_*.csv here
df_test = pd.read_csv(TEST_CSV)
print(f"Test set : {df_test.shape[0]:,} rows  "
      f"year {int(df_test.transaction_year.min())}–{int(df_test.transaction_year.max())}")
print(f"Mean price   : ${df_test.resale_price.mean():,.0f}")
print(f"Median price : ${df_test.resale_price.median():,.0f}")

Test set : 260,699 rows  year 2015–2026
Mean price   : $514,748
Median price : $480,000


## 5.2 Decode and build 25 evaluation queries

In [12]:
TOWN_COLS = [c for c in df_test.columns if c.startswith("town_")]
TYPE_COLS = [c for c in df_test.columns if c.startswith("flat_type_")]

if TOWN_COLS:
    def decode_onehot(row, cols, prefix):
        for c in cols:
            if row[c] == 1: return c.replace(prefix, "")
        return "UNKNOWN"

    df_test["town"]      = df_test.apply(lambda r: decode_onehot(r, TOWN_COLS, "town_"),      axis=1)
    df_test["flat_type"] = df_test.apply(lambda r: decode_onehot(r, TYPE_COLS, "flat_type_"), axis=1)
else:
    # Assume town and flat_type are already string columns
    pass

TOWNS       = df_test["town"].unique().tolist()
TEST_MEDIAN = int(df_test.resale_price.median())
LEASE_MIN   = int(df_test.lease_remaining_years.min())

evaluation_queries = [
    {"intent":"PRICE_ESTIMATION","query":f"How much is a 4-room flat in {TOWNS[0]} worth?",
     "ground_truth":float(df_test[(df_test.town==TOWNS[0])&(df_test.room_count==4)].resale_price.median())},
    {"intent":"PRICE_ESTIMATION","query":f"Median price of 3-room flat in {TOWNS[1]}?",
     "ground_truth":float(df_test[(df_test.town==TOWNS[1])&(df_test.room_count==3)].resale_price.median())},
    {"intent":"PRICE_ESTIMATION","query":f"Price of executive flat in {TOWNS[2]}?",
     "ground_truth":float(df_test[(df_test.town==TOWNS[2])&(df_test.flat_type=="EXECUTIVE")].resale_price.median())},
    {"intent":"PRICE_ESTIMATION","query":f"How much for a 5-room flat in {TOWNS[3]}?",
     "ground_truth":float(df_test[(df_test.town==TOWNS[3])&(df_test.room_count==5)].resale_price.median())},
    {"intent":"PRICE_ESTIMATION","query":f"Typical 4-room flat price in {TOWNS[4]}?",
     "ground_truth":float(df_test[(df_test.town==TOWNS[4])&(df_test.room_count==4)].resale_price.median())},
    {"intent":"NEIGHBOURHOOD","query":f"What amenities are near {TOWNS[0]} flats?","ground_truth":None},
    {"intent":"NEIGHBOURHOOD","query":f"Mall access from {TOWNS[1]}?","ground_truth":None},
    {"intent":"NEIGHBOURHOOD","query":f"Is {TOWNS[2]} close to MRT?","ground_truth":None},
    {"intent":"NEIGHBOURHOOD","query":f"Food courts near {TOWNS[3]}?","ground_truth":None},
    {"intent":"NEIGHBOURHOOD","query":f"Accessibility of {TOWNS[4]}?","ground_truth":None},
    {"intent":"SCHOOL_CATCHMENT","query":f"Best primary schools near {TOWNS[0]}?","ground_truth":None},
    {"intent":"SCHOOL_CATCHMENT","query":f"Best towns for schools under ${TEST_MEDIAN:,}?","ground_truth":None},
    {"intent":"SCHOOL_CATCHMENT","query":f"School quality near {TOWNS[1]} flats?","ground_truth":None},
    {"intent":"SCHOOL_CATCHMENT","query":f"Schools within 1km in {TOWNS[2]}?","ground_truth":None},
    {"intent":"SCHOOL_CATCHMENT","query":f"Primary school access in {TOWNS[3]}?","ground_truth":None},
    {"intent":"INVESTMENT_TEMPORAL","query":f"How have {TOWNS[0]} prices changed 2023-2025?","ground_truth":None},
    {"intent":"INVESTMENT_TEMPORAL","query":"Which town had highest growth in 2024?","ground_truth":None},
    {"intent":"INVESTMENT_TEMPORAL","query":f"Price trend for 4-room flats in {TOWNS[1]} since 2023?","ground_truth":None},
    {"intent":"INVESTMENT_TEMPORAL","query":"Compare 2023 vs 2025 median prices by town","ground_truth":None},
    {"intent":"INVESTMENT_TEMPORAL","query":f"Is {TOWNS[2]} appreciating faster than average?","ground_truth":None},
    {"intent":"LEASE_ADVISORY","query":f"Should I buy a flat with {LEASE_MIN+10} years lease?","ground_truth":None},
    {"intent":"LEASE_ADVISORY","query":"How does lease affect HDB prices?","ground_truth":None},
    {"intent":"LEASE_ADVISORY","query":"Are <60 year lease flats much cheaper?","ground_truth":None},
    {"intent":"LEASE_ADVISORY","query":f"70 vs 90 year lease price difference in {TOWNS[0]}?","ground_truth":None},
    {"intent":"LEASE_ADVISORY","query":"Typical discount for short-lease HDB flats?","ground_truth":None},
]
print(f"Evaluation queries: {len(evaluation_queries)}")

Evaluation queries: 25


## 5.3 Run pipeline and evaluate

In [13]:
# run_pipeline() must be available from NB04 — re-run NB04 cells first
results = []
for eq in evaluation_queries:
    try:
        out = run_pipeline(eq["query"])
        out["expected_intent"] = eq["intent"]
        out["ground_truth"]    = eq["ground_truth"]
        results.append(out)
        print(f"[OK]  {eq['intent']:<22} {eq['query'][:45]}")
    except Exception as e:
        results.append({"error": str(e), "query": eq["query"],
                        "expected_intent": eq["intent"],
                        "ground_truth": eq["ground_truth"]})
        print(f"[ERR] {eq['intent']:<22} {str(e)[:55]}")

print(f"\nCompleted: {sum(1 for r in results if 'error' not in r)}/{len(results)}")

[OK]  PRICE_ESTIMATION       How much is a 4-room flat in ANG MO KIO worth
[OK]  PRICE_ESTIMATION       Median price of 3-room flat in BEDOK?
[OK]  PRICE_ESTIMATION       Price of executive flat in BISHAN?
[OK]  PRICE_ESTIMATION       How much for a 5-room flat in BUKIT BATOK?
[OK]  PRICE_ESTIMATION       Typical 4-room flat price in BUKIT MERAH?
[OK]  NEIGHBOURHOOD          What amenities are near ANG MO KIO flats?
[OK]  NEIGHBOURHOOD          Mall access from BEDOK?
[OK]  NEIGHBOURHOOD          Is BISHAN close to MRT?
[OK]  NEIGHBOURHOOD          Food courts near BUKIT BATOK?
[OK]  NEIGHBOURHOOD          Accessibility of BUKIT MERAH?
[OK]  SCHOOL_CATCHMENT       Best primary schools near ANG MO KIO?
[OK]  SCHOOL_CATCHMENT       Best towns for schools under $480,000?
[OK]  SCHOOL_CATCHMENT       School quality near BEDOK flats?
[OK]  SCHOOL_CATCHMENT       Schools within 1km in BISHAN?
[OK]  SCHOOL_CATCHMENT       Primary school access in BUKIT BATOK?
[OK]  INVESTMENT_TEMPORAL    How 

## 5.4 Metrics — intent accuracy, RMSE%, hallucination audit

In [14]:
# Intent accuracy
correct = sum(1 for r in results if "error" not in r
              and r.get("intent") == r.get("expected_intent"))
print(f"Intent accuracy : {correct}/{len(results)} = {correct/len(results)*100:.1f}%")

# RMSE% — use % not absolute to account for 31% price drift
price_r = [r for r in results if r.get("expected_intent")=="PRICE_ESTIMATION"
           and "error" not in r and r.get("ground_truth")
           and r["answer"].get("estimate_sgd")]
if price_r:
    pct_errs = [abs(r["answer"]["estimate_sgd"] - r["ground_truth"])
                / r["ground_truth"] * 100 for r in price_r]
    rmse_pct = float(np.sqrt(np.mean(np.array(pct_errs)**2)))
    print(f"Price RMSE%     : {rmse_pct:.2f}%")

# Hallucination audit
KNOWN_FEATURES = {
    "level_mid","lease_remaining_years","floor_area_sqm","room_count",
    "dist_to_mrt_m","orientation_score","dist_to_highway_m",
    "dist_to_foodcourt_m","dist_to_nearest_mall_m","mall_count_3km",
    "mall_weighted_access_3km","dist_to_nearest_school_m","school_count_1km",
    "primary_school_quality_1km_weighted","primary_school_top_quality_1km",
    "primary_school_count_1km","resale_price","transaction_year",
}
flags = []
for r in results:
    if "error" in r or "answer" not in r: continue
    est = r["answer"].get("estimate_sgd")
    if est and not (140000 <= est <= 1700000):
        flags.append((r["query"][:45], f"estimate ${est:,} out of range"))
    for kf in r["answer"].get("key_factors", []):
        fname = kf.get("factor_name","").lower().replace(" ","_")
        if fname and fname not in KNOWN_FEATURES:
            flags.append((r["query"][:45], f"unknown factor: {fname}"))

print(f"Hallucinations  : {len(flags)}  ({'PASS' if not flags else 'FAIL — review above'})")
for q, msg in flags:
    print(f"  [{q}] {msg}")

# Final summary
print()
print("=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)
print(f"Total queries   : {len(results)}")
print(f"Pipeline errors : {sum(1 for r in results if 'error' in r)}")
print(f"Intent accuracy : {correct}/{len(results)} ({correct/len(results)*100:.1f}%)")
if price_r: print(f"Price RMSE%     : {rmse_pct:.2f}%")
print(f"Hallucinations  : {len(flags)}  ({'PASS' if not flags else 'FAIL'})")
print("=" * 50)
print("Notebook 05 complete.")

Intent accuracy : 19/25 = 76.0%
Hallucinations  : 16  (FAIL — review above)
  [How much is a 4-room flat in ANG MO KIO worth] unknown factor: flat_type
  [How much is a 4-room flat in ANG MO KIO worth] unknown factor: town
  [Price of executive flat in BISHAN?] unknown factor: flat_type
  [Typical 4-room flat price in BUKIT MERAH?] unknown factor: flat_type
  [Typical 4-room flat price in BUKIT MERAH?] unknown factor: town
  [How have ANG MO KIO prices changed 2023-2025?] unknown factor: property_type
  [How have ANG MO KIO prices changed 2023-2025?] unknown factor: town
  [How have ANG MO KIO prices changed 2023-2025?] unknown factor: floor_area
  [Price trend for 4-room flats in BEDOK since 2] unknown factor: time_period
  [Price trend for 4-room flats in BEDOK since 2] unknown factor: flat_type
  [Price trend for 4-room flats in BEDOK since 2] unknown factor: area
  [Compare 2023 vs 2025 median prices by town] unknown factor: town
  [Compare 2023 vs 2025 median prices by town] unkno